# DS605 Lab 5 — Machine Learning with Scikit-learn and From Scratch

**Dataset:** UCI Productivity Prediction of Garment Employees

This notebook covers the complete required workflow: Scikit-learn regression/classification, a NumPy/Pandas implementation from scratch, timing, evaluation, comparison, and optimization.

The assignment requires a public GitHub repository containing the runnable implementations, comparison tables, timing results, optimized manual version, and README. fileciteturn0file0L45-L49


## 1. Setup

If the dataset is not already beside this notebook, download the UCI Garment Employee Productivity CSV and set `DATA_PATH` to its location.


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TEST_SIZE = 0.20
DATA_PATH = "garments_worker_productivity.csv"


In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))


## 2. Targets and fixed split

Regression target: `actual_productivity`.

Classification target: `MeetsTarget = 1` when `actual_productivity >= targeted_productivity`, otherwise `0`. `actual_productivity` is not used as a classification input. fileciteturn0file0L10-L16


In [ ]:
TARGET_REG = "actual_productivity"
TARGETED = "targeted_productivity"

df["MeetsTarget"] = (df[TARGET_REG] >= df[TARGETED]).astype(int)

X = df.drop(columns=[TARGET_REG, "MeetsTarget"])
y_reg = df[TARGET_REG].astype(float)
y_cls = df["MeetsTarget"].astype(int)

indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_cls
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_reg, y_test_reg = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
y_train_cls, y_test_cls = y_cls.iloc[train_idx], y_cls.iloc[test_idx]

print("Train:", len(train_idx), " Test:", len(test_idx))


# Part A — Scikit-learn Implementation

The assignment asks for preprocessing of missing values, categorical variables, and scaling, followed by Linear Regression and Logistic Regression with timing and the specified metrics. fileciteturn0file0L17-L21


In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])


In [ ]:
# Scikit-learn Linear Regression
reg_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

t0 = time.perf_counter()
reg_model.fit(X_train, y_train_reg)
reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
reg_pred = reg_model.predict(X_test)
reg_pred_time = time.perf_counter() - t0

sk_reg = {
    "Implementation": "Scikit-learn",
    "MAE": mean_absolute_error(y_test_reg, reg_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, reg_pred)),
    "R2": r2_score(y_test_reg, reg_pred),
    "Training Time (s)": reg_train_time,
    "Prediction Time (s)": reg_pred_time
}
display(pd.DataFrame([sk_reg]))


In [ ]:
# Scikit-learn Logistic Regression
cls_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

t0 = time.perf_counter()
cls_model.fit(X_train, y_train_cls)
cls_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
cls_pred = cls_model.predict(X_test)
cls_pred_time = time.perf_counter() - t0

sk_cls = {
    "Implementation": "Scikit-learn",
    "Accuracy": accuracy_score(y_test_cls, cls_pred),
    "Precision": precision_score(y_test_cls, cls_pred, zero_division=0),
    "Recall": recall_score(y_test_cls, cls_pred, zero_division=0),
    "F1": f1_score(y_test_cls, cls_pred, zero_division=0),
    "Training Time (s)": cls_train_time,
    "Prediction Time (s)": cls_pred_time
}
display(pd.DataFrame([sk_cls]))


# Part B — From-Scratch NumPy/Pandas Implementation

No Scikit-learn preprocessing, model, metric, or train-test utilities are used below. The same train/test samples are reused. fileciteturn0file0L22-L29


In [ ]:
def fit_manual_preprocessor(X):
    X = X.copy()
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    state = {
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "medians": {},
        "means": {},
        "stds": {},
        "categories": {}
    }

    for c in num_cols:
        v = pd.to_numeric(X[c], errors="coerce")
        median = v.median()
        v = v.fillna(median)
        std = v.std(ddof=0)
        state["medians"][c] = median
        state["means"][c] = v.mean()
        state["stds"][c] = 1.0 if (not np.isfinite(std) or std == 0) else std

    for c in cat_cols:
        s = X[c].astype("object")
        mode = s.mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else "MISSING"
        s = s.fillna(fill).astype(str)
        state["categories"][c] = sorted(s.unique().tolist())

    return state


def transform_manual_preprocessor(X, state):
    X = X.copy()
    blocks = []

    for c in state["num_cols"]:
        v = pd.to_numeric(X[c], errors="coerce")
        v = v.fillna(state["medians"][c]).to_numpy(dtype=float)
        v = (v - state["means"][c]) / state["stds"][c]
        blocks.append(v.reshape(-1, 1))

    for c in state["cat_cols"]:
        cats = state["categories"][c]
        mapping = {cat: i for i, cat in enumerate(cats)}
        s = X[c].astype("object").fillna(cats[0]).astype(str)
        arr = np.zeros((len(X), len(cats)), dtype=float)
        for i, value in enumerate(s):
            if value in mapping:
                arr[i, mapping[value]] = 1.0
        blocks.append(arr)

    return np.hstack(blocks) if blocks else np.empty((len(X), 0))


manual_state = fit_manual_preprocessor(X_train)
Xtr = transform_manual_preprocessor(X_train, manual_state)
Xte = transform_manual_preprocessor(X_test, manual_state)

print("Manual train matrix:", Xtr.shape)
print("Manual test matrix :", Xte.shape)


In [ ]:
# Manual evaluation functions
def mae(y, p):
    return np.mean(np.abs(y - p))

def rmse(y, p):
    return np.sqrt(np.mean((y - p) ** 2))

def r2_manual(y, p):
    return 1 - np.sum((y - p) ** 2) / np.sum((y - np.mean(y)) ** 2)

def classification_metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    tp = np.sum((p == 1) & (y == 1))
    fp = np.sum((p == 1) & (y == 0))
    fn = np.sum((p == 0) & (y == 1))
    tn = np.sum((p == 0) & (y == 0))

    accuracy = (tp + tn) / len(y)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return accuracy, precision, recall, f1


def add_bias(X):
    return np.column_stack([np.ones(len(X)), X])


In [ ]:
# Manual Linear Regression — closed form
def fit_linear_closed_form(X, y, ridge=1e-8):
    Xb = add_bias(X)
    I = np.eye(Xb.shape[1])
    I[0, 0] = 0.0
    return np.linalg.solve(Xb.T @ Xb + ridge * I, Xb.T @ y)

def predict_linear(X, beta):
    return add_bias(X) @ beta

t0 = time.perf_counter()
beta = fit_linear_closed_form(Xtr, y_train_reg.to_numpy())
manual_reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
manual_reg_pred = predict_linear(Xte, beta)
manual_reg_pred_time = time.perf_counter() - t0

ytr = y_test_reg.to_numpy()

manual_reg = {
    "Implementation": "From Scratch",
    "MAE": mae(ytr, manual_reg_pred),
    "RMSE": rmse(ytr, manual_reg_pred),
    "R2": r2_manual(ytr, manual_reg_pred),
    "Training Time (s)": manual_reg_train_time,
    "Prediction Time (s)": manual_reg_pred_time
}
display(pd.DataFrame([manual_reg]))


In [ ]:
# Manual Logistic Regression — vectorized gradient descent
def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))

def fit_logistic_gd(X, y, lr=0.03, epochs=5000, l2=1e-4, tol=1e-8):
    Xb = add_bias(X)
    y = np.asarray(y, dtype=float)
    w = np.zeros(Xb.shape[1])
    previous_loss = np.inf

    for epoch in range(epochs):
        p = sigmoid(Xb @ w)
        grad = (Xb.T @ (p - y)) / len(y)
        grad[1:] += l2 * w[1:]
        w -= lr * grad

        if epoch % 50 == 0:
            loss = -np.mean(
                y * np.log(p + 1e-12) +
                (1-y) * np.log(1-p + 1e-12)
            ) + 0.5 * l2 * np.sum(w[1:] ** 2)

            if abs(previous_loss - loss) < tol:
                break
            previous_loss = loss

    return w, epoch + 1

def predict_logistic_proba(X, w):
    return sigmoid(add_bias(X) @ w)

def predict_logistic(X, w, threshold=0.5):
    return (predict_logistic_proba(X, w) >= threshold).astype(int)

t0 = time.perf_counter()
w, epochs_used = fit_logistic_gd(
    Xtr, y_train_cls.to_numpy(),
    lr=0.03, epochs=5000, l2=1e-4
)
manual_cls_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
manual_cls_pred = predict_logistic(Xte, w)
manual_cls_pred_time = time.perf_counter() - t0

acc, prec, rec, f1 = classification_metrics(
    y_test_cls.to_numpy(), manual_cls_pred
)

manual_cls = {
    "Implementation": "From Scratch",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1": f1,
    "Training Time (s)": manual_cls_train_time,
    "Prediction Time (s)": manual_cls_pred_time,
    "Epochs": epochs_used
}
display(pd.DataFrame([manual_cls]))


# Part C — Comparison and Optimization

The assignment asks for compact comparison tables, timing, and an optimized manual implementation. fileciteturn0file0L30-L36


In [ ]:
reg_comparison = pd.DataFrame([sk_reg, manual_reg])
display(reg_comparison)

cls_comparison = pd.DataFrame([
    sk_cls,
    {k: v for k, v in manual_cls.items() if k != "Epochs"}
])
display(cls_comparison)


In [ ]:
# Simple manual logistic optimization experiment
configs = [
    {"lr": 0.01, "epochs": 3000, "l2": 1e-4},
    {"lr": 0.03, "epochs": 5000, "l2": 1e-4},
    {"lr": 0.05, "epochs": 5000, "l2": 1e-4},
    {"lr": 0.03, "epochs": 7000, "l2": 1e-3},
]

rows = []
for cfg in configs:
    t0 = time.perf_counter()
    w_try, ep = fit_logistic_gd(Xtr, y_train_cls.to_numpy(), **cfg)
    train_time = time.perf_counter() - t0

    pred = predict_logistic(Xte, w_try)
    acc, prec, rec, f1 = classification_metrics(
        y_test_cls.to_numpy(), pred
    )

    rows.append({
        **cfg,
        "Epochs Used": ep,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "Training Time (s)": train_time
    })

optimization_results = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(optimization_results)


In [ ]:
best = optimization_results.iloc[0]
best_lr = float(best["lr"])
best_epochs = int(best["epochs"])
best_l2 = float(best["l2"])

t0 = time.perf_counter()
w_opt, opt_epochs_used = fit_logistic_gd(
    Xtr, y_train_cls.to_numpy(),
    lr=best_lr, epochs=best_epochs, l2=best_l2
)
opt_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
opt_pred = predict_logistic(Xte, w_opt)
opt_pred_time = time.perf_counter() - t0

opt_acc, opt_prec, opt_rec, opt_f1 = classification_metrics(
    y_test_cls.to_numpy(), opt_pred
)

optimized_manual = pd.DataFrame([{
    "Implementation": "Optimized From Scratch",
    "Accuracy": opt_acc,
    "Precision": opt_prec,
    "Recall": opt_rec,
    "F1": opt_f1,
    "Training Time (s)": opt_train_time,
    "Prediction Time (s)": opt_pred_time,
    "Learning Rate": best_lr,
    "L2": best_l2,
    "Epochs": opt_epochs_used
}])

display(optimized_manual)


## 15. Regression visualization

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_reg, reg_pred, alpha=0.65)
lo = min(y_test_reg.min(), reg_pred.min())
hi = max(y_test_reg.max(), reg_pred.max())
plt.plot([lo, hi], [lo, hi], "--")
plt.xlabel("Actual Productivity")
plt.ylabel("Predicted Productivity")
plt.title("Scikit-learn Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()


## 16. Submission checklist

Before pushing to GitHub:

- [ ] This notebook runs from start to finish.
- [ ] Dataset file is available through the documented download/source instructions.
- [ ] Scikit-learn regression and classification are included.
- [ ] Manual NumPy/Pandas preprocessing and models are included.
- [ ] The same fixed train/test split is reused.
- [ ] Regression metrics: MAE, RMSE, R².
- [ ] Classification metrics: accuracy, precision, recall, F1.
- [ ] Training and prediction times are recorded.
- [ ] Comparison tables are included.
- [ ] Manual optimization results are included.
- [ ] README explains setup, execution, observations, and limitations.
- [ ] Repository is public.

The official assignment states that the final submission is **the public GitHub repository link only**. fileciteturn0file0L45-L49
